# 2024년 10월 17일 GTX-A 광역버스 교통카드 합성데이터 조회

33개 버스번호를 기준으로 공공데이터포털 API를 호출한다.

- `opr_ymd`: `20241017`
- `users_type_cd`: `01`
- `ride_ctpv_cd`: `41`
- `rte_id`: CSV의 버스번호(`1000`, `M7111` 등)

In [ ]:
from pathlib import Path
from getpass import getpass
import time
import requests
import pandas as pd

BASE_URL = "https://apis.data.go.kr/1613000/RegionalTransportationCardUsageSyntheticData/getGyeonggiTransportationCardUsageSyntheticData"
SERVICE_KEY = getpass("공공데이터포털 인증키를 입력하세요: ")
OPR_YMD = "20241017"
USERS_TYPE_CD = "01"
RIDE_CTPV_CD = "41"
PAGE_SIZE = 1000

print("설정 완료")

In [ ]:
# 33개 노선번호 읽기
route_file = Path("gtx_a_seoul_bus_outputs/gtx_a_north_33_unique_routes.csv")
routes = pd.read_csv(route_file, dtype=str, encoding="utf-8-sig")

# CSV에 표시용으로 붙어 있는 앞쪽 작은따옴표 제거
routes["버스번호"] = routes["버스번호"].str.strip().str.lstrip("'")
routes = routes.drop_duplicates(subset=["버스번호"]).reset_index(drop=True)

print(f"조회할 노선 수: {len(routes)}")
display(routes[["버스번호", "GTX_A_역", "대상지역", "노선ID"]])

In [ ]:
def parse_response(payload):
    # API 응답이 response로 한 번 감싸지는 경우와
    # header/body가 바로 오는 경우를 모두 처리
    response = payload.get("response", payload.get("Response", payload))
    header = response.get("header", {}) or {}
    body = response.get("body", {}) or {}
    items = body.get("items", {}) or {}
    items = items.get("item", []) if isinstance(items, dict) else items
    if isinstance(items, dict):
        items = [items]
    if not isinstance(items, list):
        items = []
    return items, header, body

def request_route(route_no, page_no=1):
    params = {
        "serviceKey": SERVICE_KEY,
        "pageNo": page_no,
        "numOfRows": PAGE_SIZE,
        "dataType": "JSON",
        "opr_ymd": OPR_YMD,
        "rte_id": route_no,
        "users_type_cd": USERS_TYPE_CD,
        "ride_ctpv_cd": RIDE_CTPV_CD,
    }
    r = requests.get(BASE_URL, params=params, timeout=60)
    print("요청 URL(인증키 제외):", r.url.split("serviceKey=")[0] + "serviceKey=***")
    print("HTTP 상태코드:", r.status_code)
    r.raise_for_status()
    return parse_response(r.json())

In [ ]:
# 먼저 1000번 노선의 STCIS 노선 ID 41016025를 1페이지 시험 조회
# 조회일자: 2024년 10월 17일 (OPR_YMD = 20241017)
test_items, test_header, test_body = request_route("41016025", page_no=1)
print("header:", test_header)
print("body 주요 키:", list(test_body.keys()))
print("조회 건수:", len(test_items))
if test_items:
    display(pd.DataFrame(test_items).head())

In [ ]:
# 33개 노선 전체 조회
all_rows = []
results = []

for i, route_no in enumerate(routes["버스번호"], start=1):
    route_rows = []
    try:
        page_no = 1
        while True:
            items, header, body = request_route(route_no, page_no=page_no)
            route_rows.extend(items)
            total = int(body.get("totalCount", body.get("totalcount", 0)) or 0)
            if not items or (total and len(route_rows) >= total) or len(items) < PAGE_SIZE:
                break
            page_no += 1
        for row in route_rows:
            row["조회_버스번호"] = route_no
        all_rows.extend(route_rows)
        results.append({"버스번호": route_no, "상태": "성공", "건수": len(route_rows), "오류": ""})
        print(f"[{i}/{len(routes)}] {route_no}: {len(route_rows)}건")
    except Exception as e:
        results.append({"버스번호": route_no, "상태": "실패", "건수": 0, "오류": repr(e)})
        print(f"[{i}/{len(routes)}] {route_no}: 실패 - {e}")
    time.sleep(0.2)

result_df = pd.DataFrame(results)
print("전체 API 응답 행 수:", len(all_rows))
display(result_df)

In [ ]:
# 원자료 저장 및 노선별 이용량 집계
out_dir = Path("gtx_a_seoul_bus_outputs")
out_dir.mkdir(exist_ok=True)

raw_df = pd.DataFrame(all_rows)
raw_path = out_dir / "gtx_a_bus_usage_20241017_raw.csv"
status_path = out_dir / "gtx_a_bus_usage_20241017_query_status.csv"
raw_df.to_csv(raw_path, index=False, encoding="utf-8-sig")
result_df.to_csv(status_path, index=False, encoding="utf-8-sig")

if not raw_df.empty and "utztn_nope" in raw_df.columns:
    raw_df["utztn_nope"] = pd.to_numeric(raw_df["utztn_nope"], errors="coerce").fillna(0)
    summary = (raw_df.groupby("조회_버스번호", as_index=False)
               .agg(이용건수=("utztn_nope", "sum"), API응답행수=("조회_버스번호", "size")))
else:
    summary = result_df.rename(columns={"버스번호": "조회_버스번호", "건수": "API응답행수"})

summary = routes.merge(summary, left_on="버스번호", right_on="조회_버스번호", how="left")
summary_path = out_dir / "gtx_a_bus_usage_20241017_by_route.csv"
summary.to_csv(summary_path, index=False, encoding="utf-8-sig")

print("원자료 저장:", raw_path)
print("조회 상태 저장:", status_path)
print("노선별 집계 저장:", summary_path)
display(summary)

## 2025년 10월 16일 조회: 최종 32개 노선

7200번은 폐선으로 제외하고, STCIS에서 확인한 32개 노선 ID를 기준으로 조회합니다.

In [ ]:
# 최종 분석 대상 32개 노선의 STCIS 노선 ID
TARGET_DATE = "20251016"
OPR_YMD = TARGET_DATE
route_id_map_2025 = {
    "41016025": "1000", "41016018": "1082", "41016045": "1100",
    "41016027": "1200", "41016044": "1500", "41016055": "1500(예약)",
    "41016042": "1900", "41084003": "2200", "41084009": "2200-1",
    "41084007": "3100", "41084006": "3100N", "41016049": "3400",
    "41011156": "6701", "41016050": "7101", "41084008": "7111",
    "41016903": "M7111", "41016904": "M7111(예약)", "41084903": "M7154",
    "41003902": "M7412", "41003910": "M7412(예약)", "41084001": "G7426",
    "41165001": "7602", "41084002": "G7625", "41016902": "M7731",
    "41084004": "9030", "41084005": "9030-1", "41003717": "9600",
    "41016029": "9700", "41034153": "9709", "41034156": "9709N",
    "41034123": "9710", "41034145": "9710-1",
}

all_rows_20251016 = []
results_20251016 = []

for i, (route_id, route_no) in enumerate(route_id_map_2025.items(), start=1):
    route_rows = []
    try:
        page_no = 1
        while True:
            items, header, body = request_route(route_id, page_no=page_no)
            route_rows.extend(items)
            total = int(body.get("totalCount", body.get("totalcount", 0)) or 0)
            if not items or (total and len(route_rows) >= total) or len(items) < PAGE_SIZE:
                break
            page_no += 1
        for row in route_rows:
            row["조회_노선ID"] = route_id
            row["조회_버스번호"] = route_no
            row["조회일자"] = TARGET_DATE
        all_rows_20251016.extend(route_rows)
        results_20251016.append({"노선ID": route_id, "버스번호": route_no, "상태": "성공", "행수": len(route_rows), "오류": ""})
        print(f"[{i}/{len(route_id_map_2025)}] {route_no} ({route_id}): {len(route_rows)}건")
    except Exception as e:
        results_20251016.append({"노선ID": route_id, "버스번호": route_no, "상태": "실패", "행수": 0, "오류": repr(e)})
        print(f"[{i}/{len(route_id_map_2025)}] {route_no} ({route_id}): 실패 - {e}")
    time.sleep(0.2)

result_df_20251016 = pd.DataFrame(results_20251016)
raw_df_20251016 = pd.DataFrame(all_rows_20251016)
out_dir = Path("gtx_a_seoul_bus_outputs")
out_dir.mkdir(exist_ok=True)
raw_path_20251016 = out_dir / "gtx_a_bus_usage_20251016_raw.csv"
status_path_20251016 = out_dir / "gtx_a_bus_usage_20251016_query_status.csv"
raw_df_20251016.to_csv(raw_path_20251016, index=False, encoding="utf-8-sig")
result_df_20251016.to_csv(status_path_20251016, index=False, encoding="utf-8-sig")

if not raw_df_20251016.empty and "utztn_nope" in raw_df_20251016.columns:
    raw_df_20251016["utztn_nope"] = pd.to_numeric(raw_df_20251016["utztn_nope"], errors="coerce").fillna(0)
    summary_20251016 = (raw_df_20251016.groupby(["조회_노선ID", "조회_버스번호"], as_index=False)
        .agg(이용건수=("utztn_nope", "sum"), API응답행수=("조회_노선ID", "size")))
else:
    summary_20251016 = result_df_20251016.rename(columns={"행수": "API응답행수"})

summary_path_20251016 = out_dir / "gtx_a_bus_usage_20251016_by_route.csv"
summary_20251016.to_csv(summary_path_20251016, index=False, encoding="utf-8-sig")
print("전체 API 응답 행 수:", len(raw_df_20251016))
print("원자료 저장:", raw_path_20251016)
print("조회 상태 저장:", status_path_20251016)
print("노선별 집계 저장:", summary_path_20251016)
display(result_df_20251016)
display(summary_20251016)